In [4]:
## Prepared by Muhammad Izzul Danish SW01083596
## Prepared by Mohd Adli Syukri SW01083745

import pandas as pd
import re
import emoji
import string
import nltk
from bs4 import BeautifulSoup
from autocorrect import Speller
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')  # For lemmatization
nltk.download('omw-1.4')  # WordNet lexical database
nltk.download('averaged_perceptron_tagger_eng')  # For POS tagging
nltk.download('punkt_tab')  # For tokenization

# Initialize tools
spell = Speller(lang='en')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Dictionary of slang words and their replacements
slang_dict = {
    "tbh": "to be honest",
    "omg": "oh my god",
    "lol": "laugh out loud",
    "idk": "I don't know",
    "brb": "be right back",
    "btw": "by the way",
    "imo": "in my opinion",
    "smh": "shaking my head",
    "fyi": "for your information",
    "np": "no problem",
    "ikr": "I know right",
    "asap": "as soon as possible",
    "bff": "best friend forever",
    "gg": "good game",
    "hmu": "hit me up",
    "rofl": "rolling on the floor laughing"
}

# Contractions dictionary
contractions_dict = {
    "wasn't": "was not",
    "isn't": "is not",
    "aren't": "are not",
    "weren't": "were not",
    "doesn't": "does not",
    "don't": "do not",
    "didn't": "did not",
    "can't": "cannot",
    "couldn't": "could not",
    "shouldn't": "should not",
    "wouldn't": "would not",
    "won't": "will not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "i'm": "i am",
    "you're": "you are",
    "he's": "he is",
    "she's": "she is",
    "it's": "it is",
    "we're": "we are",
    "they're": "they are",
    "i've": "i have",
    "you've": "you have",
    "we've": "we have",
    "they've": "they have",
    "i'd": "i would",
    "you'd": "you would",
    "he'd": "he would",
    "she'd": "she would",
    "we'd": "we would",
    "they'd": "they would",
    "i'll": "i will",
    "you'll": "you will",
    "he'll": "he will",
    "she'll": "she will",
    "we'll": "we will",
    "they'll": "they will",
    "let's": "let us",
    "that's": "that is",
    "who's": "who is",
    "what's": "what is",
    "where's": "where is",
    "when's": "when is",
    "why's": "why is"
}

# Remove any URLs that start with "http" or "www" from the text
def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

# extracts only the text, removing all HTML tags
def remove_html(text):
    return BeautifulSoup(text, "html.parser").get_text()

# replace emoji with ''
def remove_emojis(text):
    return emoji.replace_emoji(text, replace='')

# Replace internet slang/chat words
def replace_slang(text):
    escaped_slang_words = []
    for word in slang_dict.keys():
        escaped_word = re.escape(word)
        escaped_slang_words.append(escaped_word)

    slang_pattern = r'\b(' + '|'.join(escaped_slang_words) + r')\b'

    def replace_match(match):
        slang_word = match.group(0)
        return slang_dict[slang_word.lower()]

    replaced_text = re.sub(slang_pattern, replace_match, text, flags=re.IGNORECASE)
    return replaced_text

# Function to expand contractions
escaped_contractions = []
for contraction in contractions_dict.keys():
    escaped_contraction = re.escape(contraction)
    escaped_contractions.append(escaped_contraction)

joined_contractions = "|".join(escaped_contractions)
contractions_pattern = r'\b(' + joined_contractions + r')\b'
compiled_pattern = re.compile(contractions_pattern, flags=re.IGNORECASE)

def replace_contractions(text):
    def replace_match(match):
        matched_word = match.group(0)
        lower_matched_word = matched_word.lower()
        expanded_form = contractions_dict[lower_matched_word]
        return expanded_form

    expanded_text = compiled_pattern.sub(replace_match, text)
    return expanded_text

# Function to remove punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

# Function to remove numbers
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

# Function to correct spelling using AutoCorrect
def correct_spelling(text):
    return spell(text)

# Function to remove stopwords
def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return " ".join(filtered_words)

# Function to map NLTK POS tags to WordNet POS tags
def get_wordnet_pos(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# Function to lemmatize text with POS tagging
def lemmatize_text(text):
    if not isinstance(text, str):
        return ""

    words = word_tokenize(text)
    pos_tags = pos_tag(words)

    lemmatized_words = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in pos_tags
    ]

    return " ".join(lemmatized_words)

# Function to tokenize text
def tokenize_text(text):
    if not isinstance(text, str):
        return []
    return word_tokenize(text)

# Function to apply all preprocessing steps
def preprocess_text(text):
    text = text.lower()  # Step 1: Lowercasing
    text = remove_urls(text)  # Step 2: Remove URLs
    text = remove_html(text)  # Step 3: Remove HTML tags
    text = remove_emojis(text)  # Step 4: Remove Emojis
    text = replace_slang(text)  # Step 5: Replace Slang
    text = replace_contractions(text)  # Step 6: Expand Contractions
    text = remove_punctuation(text)  # Step 7: Remove Punctuation
    text = remove_numbers(text)  # Step 8: Remove Numbers
    text = correct_spelling(text)  # Step 9: Correct Spelling
    text = remove_stopwords(text)  # Step 10: Remove Stopwords
    text = lemmatize_text(text)  # Step 11: Lemmatization
    text = tokenize_text(text)  # Step 12: Tokenization
    return text

# Load dataset
df = pd.read_csv("UNITENReview.csv", encoding="latin1")  # Replace with your file

# Apply preprocessing pipeline
df["processed"] = df["Review"].apply(preprocess_text)

df.to_csv("Result.csv", index=False)
# Display the first few rows
print(df[["Review", "processed"]].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\IM11\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\IM11\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\IM11\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\IM11\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\IM11\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
C:\Users\IM11\AppData\Local\Temp\ipykernel_20496\2186480371.py:103: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You

                                              Review  \
0  Im happy with uniten actually, even the people...   
1  Iâm having a pretty good time here, happy to...   
2        a very neutral place in terms of everything   
3  I would say Uniten it's  a good university  bu...   
4   UNITEN is well-regarded, particularly for its...   

                                           processed  
0      [im, happy, unite, actually, even, people, w]  
1  [iâm, pretty, good, time, happy, meet, w, pe...  
2                 [neutral, place, term, everything]  
3  [would, say, united, good, university, issue, ...  
4  [united, wellregarded, particularly, strong, e...  


In [5]:
import json
import csv
import time
import random
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Any
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import requests
from bs4 import BeautifulSoup

# Target product page that contains the Judge.me reviews widget
PRODUCT_URL = "https://tudungpeople.com/products/hijab-ring?variant=44400372154517"


# ----------------------------
# Helpers (polite scraping)
# ----------------------------
def polite_sleep(a: float = 1.0, b: float = 2.5):
    """
    Sleep for a random time between 'a' and 'b' seconds.
    Purpose: reduce request burstiness and behave more like a normal user.
    """
    time.sleep(random.uniform(a, b))


def robots_allows(url: str, user_agent: str = "*") -> bool:
    """
    Check robots.txt to see whether we are allowed to fetch the given URL.

    Note:
    - robots.txt is not a security feature, but it is a site rule.
    - If robots.txt fetch fails (network issue), this function returns True
      (meaning you choose the risk).
    """
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

    rp = RobotFileParser()
    rp.set_url(robots_url)

    try:
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception:
        # If we cannot read robots.txt, we do not hard-block automatically.
        return True


def hard_block_sensitive_paths(url: str) -> None:
    """
    Hard-block sensitive Shopify paths to avoid scraping areas that may involve
    accounts, carts, checkouts, admin pages, or orders.

    This is a safety/ethics guardrail.
    """
    blocked = ["/checkout", "/checkouts", "/cart", "/carts", "/account", "/admin", "/orders"]
    path = urlparse(url).path.lower()

    # If URL path starts with any blocked prefix, stop execution immediately
    if any(path.startswith(b) for b in blocked):
        raise ValueError(f"Blocked URL path for safety/ToS reasons: {path}")


def fetch_html(url: str, session: requests.Session) -> str:
    """
    Download the HTML of a page using a requests Session.

    Why Session:
    - Reuses connections (faster)
    - Keeps cookies if the site sets any (more realistic browsing behavior)

    Raises:
    - HTTP errors via raise_for_status()
    """
    headers = {
        # Identify the client; some sites block default Python user agents
        "User-Agent": "Mozilla/5.0 (compatible; PublicReviewsScraper/1.0)",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    }

    r = session.get(url, headers=headers, timeout=25)
    r.raise_for_status()
    return r.text


# ----------------------------
# Review parsing
# ----------------------------
@dataclass
class Review:
    """
    Data model for one review extracted from the HTML.
    Using dataclass makes it easy to store structured fields and convert to dict.
    """
    review_id: Optional[str]
    reviewer_name: Optional[str]
    location: Optional[str]
    rating: Optional[int]
    created_at: Optional[str]
    title: Optional[str]
    body: Optional[str]
    product_title: Optional[str]
    product_url: Optional[str]
    verified_buyer: Optional[bool]
    pictures: List[str]


def parse_reviews_from_html(html: str) -> Dict[str, Any]:
    """
    Parse the Judge.me widget and review blocks from the HTML.

    Returns a dict with:
    - summary (average rating, total review count, updated time, image url)
    - reviews (list of dicts)
    - review_count_parsed (how many reviews are on this HTML page)
    - has_widget (True/False)
    """
    # Parse HTML into a DOM-like structure
    soup = BeautifulSoup(html, "lxml")

    # Find the Judge.me widget container (holds summary stats in data-* attributes)
    widg = soup.select_one(".jdgm-rev-widg")

    # Extract widget-level summary info if widget exists
    summary = {}
    if widg:
        summary = {
            "average_rating": float(widg.get("data-average-rating")) if widg.get("data-average-rating") else None,
            "number_of_reviews": int(widg.get("data-number-of-reviews")) if widg.get("data-number-of-reviews") else None,
            "updated_at": widg.get("data-updated-at"),
            "image_url": widg.get("data-image-url"),
        }

    reviews: List[Review] = []

    # Each review is typically inside an element with class ".jdgm-rev"
    for r in soup.select(".jdgm-rev"):
        # Extract rating from data-score attribute (if present)
        rating = None
        rating_el = r.select_one(".jdgm-rev__rating")
        if rating_el and rating_el.get("data-score"):
            try:
                rating = int(rating_el["data-score"])
            except ValueError:
                rating = None  # In case data-score is not a valid integer

        # Extract timestamp (Judge.me often stores the date in a data-content attribute)
        ts_el = r.select_one(".jdgm-rev__timestamp")
        created_at = ts_el.get("data-content") if ts_el else None

        # Extract all picture links attached to the review (if any)
        pics = [a["href"] for a in r.select(".jdgm-rev__pic-link[href]")]

        # Extract verified buyer flag from data-verified-buyer attribute
        verified_attr = r.get("data-verified-buyer")
        verified = True if verified_attr == "true" else (False if verified_attr == "false" else None)

        # Build Review object (safe-get text only if element exists)
        reviews.append(
            Review(
                review_id=r.get("data-review-id"),
                reviewer_name=(
                    r.select_one(".jdgm-rev__author").get_text(strip=True)
                    if r.select_one(".jdgm-rev__author") else None
                ),
                location=(
                    r.select_one(".jdgm-rev__location").get_text(" ", strip=True)
                    if r.select_one(".jdgm-rev__location") else None
                ),
                rating=rating,
                created_at=created_at,
                title=(
                    r.select_one(".jdgm-rev__title").get_text(" ", strip=True)
                    if r.select_one(".jdgm-rev__title") else None
                ),
                body=(
                    r.select_one(".jdgm-rev__body").get_text(" ", strip=True)
                    if r.select_one(".jdgm-rev__body") else None
                ),
                product_title=r.get("data-product-title"),
                product_url=r.get("data-product-url"),
                verified_buyer=verified,
                pictures=pics,
            )
        )

    # Convert Review dataclasses into dicts so they can be saved as JSON/CSV
    return {
        "summary": summary,
        "reviews": [asdict(x) for x in reviews],
        "review_count_parsed": len(reviews),
        "has_widget": bool(widg),
    }


def save_json(data: Dict[str, Any], path: str):
    """
    Save parsed data into a JSON file with UTF-8 encoding.
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def save_csv(reviews: List[Dict[str, Any]], path: str):
    """
    Save reviews into CSV.
    - fieldnames are the union of all keys (sorted)
    - pictures list is flattened into a single string joined by ' | '
    """
    if not reviews:
        return

    # Collect all possible keys across all reviews so CSV columns are consistent
    fieldnames = sorted({k for r in reviews for k in r.keys()})

    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

        for r in reviews:
            row = dict(r)

            # Convert list of picture URLs into a single cell string
            if isinstance(row.get("pictures"), list):
                row["pictures"] = " | ".join(row["pictures"])

            w.writerow(row)


def main():
    """
    Main workflow:
    1) Block unsafe URL paths
    2) Check robots.txt permission
    3) Fetch HTML
    4) Parse reviews from HTML
    5) Save JSON + CSV outputs
    """
    # Safety guard: avoid sensitive Shopify paths
    hard_block_sensitive_paths(PRODUCT_URL)

    # Respect robots.txt if it disallows scraping
    if not robots_allows(PRODUCT_URL):
        raise RuntimeError("robots.txt likely disallows this URL for your user-agent. Don't scrape it.")

    # Create a session for efficiency
    session = requests.Session()

    # Sleep once before the request to reduce aggressive behavior
    polite_sleep()

    # Fetch HTML and parse it
    html = fetch_html(PRODUCT_URL, session)
    data = parse_reviews_from_html(html)

    # Print summary info to console (quick sanity check)
    print("Has Judge.me widget:", data["has_widget"])
    print("Parsed reviews on page:", data["review_count_parsed"])
    print("Summary:", data["summary"])

    # Save outputs to files
    save_json(data, "reviews_from_html.json")
    save_csv(data["reviews"], "reviews_from_html.csv")
    print("Saved: reviews_from_html.json + reviews_from_html.csv")


# Standard Python entry point guard
if __name__ == "__main__":
    main()

Has Judge.me widget: True
Parsed reviews on page: 8
Summary: {'average_rating': 4.94, 'number_of_reviews': 33, 'updated_at': '2026-02-19T12:23:06Z', 'image_url': 'https://cdn.shopify.com/s/files/1/0260/2821/2269/files/TudungPeople-Hijab-Ring-Campaign-00.jpg?v=1710991151'}
Saved: reviews_from_html.json + reviews_from_html.csv
